HMM -> Hidden Markov Model

In [1]:
!git clone https://github.com/UniversalDependencies/UD_Spanish-AnCora.git

Cloning into 'UD_Spanish-AnCora'...
remote: Enumerating objects: 1394, done.
remote: Counting objects: 100% (3/3), done.
remote: Compressing objects: 100% (3/3), done.
remote: Total 1394 (delta 0), reused 0 (delta 0), pack-reused 1391 (from 3)
Receiving objects: 100% (1394/1394), 445.40 MiB | 15.32 MiB/s, done.
Resolving deltas: 100% (1002/1002), done.


In [2]:
!pip install conllu

In [3]:
from conllu import parse_incr

In [4]:
data_file = open("/content/UD_Spanish-AnCora/es_ancora-ud-dev.conllu", "r", encoding="utf-8")

In [5]:
for tokenlist in parse_incr(data_file):
  print(tokenlist.serialize())

Se han truncado las últimas 5000 líneas del flujo de salida.
36	vez	vez	NOUN	ncfs000	Gender=Fem|Number=Sing	33	obl	33:obl	SpaceAfter=No|ArgTem=argM:adv
37	,	,	PUNCT	fc	PunctType=Comm	36	punct	36:punct	_
38	17	17	NUM	_	NumForm=Digit|NumType=Card	40	nummod	40:nummod	Entity=(CESSCASTP20011002160c16--3-gstype:gen
39	nuevos	nuevo	ADJ	aq0mp0	Gender=Masc|Number=Plur	40	amod	40:amod	_
40	fichajes	fichaje	NOUN	ncmp000	Gender=Masc|Number=Plur	33	obj	33:obj	SpaceAfter=No|ArgTem=arg1:pat
41	,	,	PUNCT	fc	PunctType=Comm	45	punct	45:punct	_
42	entre	entre	ADP	sps00	_	43	case	43:case	_
43	ellos	él	PRON	pp3mp000	Case=Acc,Nom|Gender=Masc|Number=Plur|Person=3|PronType=Prs	45	nmod	45:nmod	Entity=(CESSCASTP20011002160c16--1-CorefType:ident,gstype:gen)
44	el	el	DET	da0ms0	Definite=Def|Gender=Masc|Number=Sing|PronType=Art	45	det	45:det	Entity=(NOCOREF:Spec.person-person-2-gstype:spec
45	portugués	portugués	NOUN	ncms000	Gender=Masc|Number=Sing	40	appos	40:appos	_
46	Conceiçao	Conceiçao	PROPN	np00000	_	45	appo

In [6]:
tokenlist[1]

{'id': 2,
 'form': 'El',
 'lemma': 'el',
 'upos': 'DET',
 'xpos': 'da0ms0',
 'feats': {'Definite': 'Def',
  'Gender': 'Masc',
  'Number': 'Sing',
  'PronType': 'Art'},
 'head': 5,
 'deprel': 'det',
 'deps': [('det', 5)],
 'misc': {'Entity': '(NOCOREF:Gen--1-gstype:gen'}}

In [7]:
tokenlist[1]["form"] + "|" + tokenlist[1]["upos"]

'El|DET'

# Entrenamiento del modelo

In [8]:
data_file = open("/content/UD_Spanish-AnCora/es_ancora-ud-dev.conllu", "r", encoding="utf-8")

tagCountDict = {} # Contador de etiquetas
emissionDict = {} # Contar las emisiones (palabra|etiqueta)
transitionDict = {} # Contar las transiciones entre etiquetas

prev_tag = None

for tokenlist in parse_incr(data_file): # iteramos sobre oraciones
  for token in tokenlist: # iteramos sobre las palabras de la oración

    # 1. Conteo de tags
    if token["upos"] in tagCountDict.keys():
      tagCountDict.update({token["upos"]: tagCountDict[token["upos"]] + 1})
    else:
      tagCountDict.update({token["upos"]: 1})

    # 2. Conteo de emisiones
    word_tag = token["form"].lower() + "|" + token["upos"]
    if word_tag in emissionDict.keys():
      emissionDict.update({word_tag: emissionDict[word_tag] + 1})
    else:
      emissionDict.update({word_tag: 1})

    if prev_tag is None:
      prev_tag = token["upos"]

    # 3. Conteo de transiciones
    transition_tag = token["upos"] + "|" + prev_tag
    if transition_tag in transitionDict.keys():
      transitionDict.update({transition_tag: transitionDict[transition_tag] + 1})
    else:
      transitionDict.update({transition_tag: 1})

    prev_tag = token["upos"]

print(f"tags: {len(tagCountDict)} | emisiones: {len(emissionDict)} | transiciones: {len(transitionDict)}")

tags: 17 | emisiones: 10516 | transiciones: 214


## Cálculo de Probabilidades

1. Probabilidad de transición:
P(tag|prevtag) = C(prevtag, tag)/C(prevtag)

- tagCountDict['NOUN'] = 1000 -> NOUN aparece 1000 veces
- transitionDict['PART|NOUN'] = 800 -> despues de NOUN viene PART 800 veces

P('PART|NOUN') = 800/1000 -> 80% de las veces despues de un NOUN viene un PART


In [9]:
transitionProbDict = {} # probabilidades de transición
emissionProbDict = {}

for key in transitionDict.keys():
  tag, prev_tag = key.split("|")
  if tagCountDict[prev_tag] > 0:
    transitionProbDict.update({key: transitionDict[key]/tagCountDict[prev_tag]})
  else:
    print(f"key: {key} | zero division" )

for key in emissionDict.keys():
  word, tag = key.split("|")
  if emissionDict[key] > 0:
    emissionProbDict.update({key: emissionDict[key]/tagCountDict[tag]})
  else:
    print(f"key: {key} | zero division" )

print(f"Prob transiciones: {len(transitionProbDict)} | Prob Emisiones: {len(emissionProbDict)}")

Prob transiciones: 214 | Prob Emisiones: 10516


In [10]:
import numpy as np
np.save("transitionHMM.npy", transitionProbDict)
np.save("emissionHMM.npy", emissionProbDict)

In [12]:
transition_prob_dict = np.load("/content/transitionHMM.npy", allow_pickle=True).item()
emission_prob_dict = np.load("/content/emissionHMM.npy", allow_pickle=True).item()

In [13]:
emission_prob_dict

{'el|DET': 0.3427614439895795,
 'gobernante|NOUN': 0.00020848535390388824,
 ',|PUNCT': 0.45137236236712674,
 'con|ADP': 0.051977003402557787,
 'ganada|ADJ': 0.0002824060999717594,
 'fama|NOUN': 0.00010424267695194412,
 'desde|ADP': 0.008799718409010912,
 'que|SCONJ': 0.6382042253521126,
 'llegó|VERB': 0.0021915406530791147,
 'hace|VERB': 0.00898531667762437,
 '16|NUM': 0.011956521739130435,
 'meses|NOUN': 0.0028145522777024913,
 'al|_': 0.2765451664025357,
 'a|ADP': 0.15205913410770855,
 'poder|NOUN': 0.0011466694464713854,
 'de|ADP': 0.4614572333685322,
 'explotar|VERB': 0.00043830813061582295,
 'máximo|NOUN': 0.00020848535390388824,
 'su|DET': 0.043418930653765044,
 'oratoria|NOUN': 0.00010424267695194412,
 'y|CCONJ': 0.7771664374140302,
 'acusado|ADJ': 0.0008472182999152782,
 'por|ADP': 0.05972075560248739,
 'sus|DET': 0.01724351817392383,
 'detractores|NOUN': 0.0003127280308558324,
 'incontinencia|NOUN': 0.00010424267695194412,
 'verbal|ADJ': 0.0005648121999435188,
 'enmudeció|VERB